# 01 — Детерминированное ядро ассистента бухгалтера

LLM-шаги замоканы (`ScriptedLLM`), GPU не нужен. Показываем write-path (память), read-path (контекст), проверку соответствия и оценку на dev/golden наборах.

In [ ]:
import os, sys
if not os.path.exists("src/acctltm"):
    !git clone https://github.com/serge3emskov/accountant-llm-assistant.git
    os.chdir("accountant-llm-assistant")
!pip install -q pydantic rapidfuzz
sys.path.insert(0, "src")

In [ ]:
from acctltm import (AccountantAssistant, MemoryStore, Normalizer, AccountingCatalog,
                     ComplianceChecker)
from acctltm.llm import ScriptedLLM
from acctltm.devcheck import load_jsonl, run_devset
from acctltm.evaluation import run_golden, eval_query_classifier
from acctltm.synthetic import load_bank_statement
import json

## Справочник

Операции, план счетов и синтетический реестр контрагентов.

In [ ]:
cat = AccountingCatalog()
print("контрагентов в реестре:", len(cat.counterparties))
print("resolve('Ромашка'):", cat.resolve("Ромашка").label)
print("resolve('расчётный счёт'):", cat.resolve("расчётный счёт").label)
print("упоминания:", [e.label for e in cat.mentions("Оплатили Ромашке наличными за товары")])

## Сквозной диалог (LLM замоканы)

Бухгалтер рассказывает о компании, затем задаёт вопрос. Код проверяет операции по профилю.

In [ ]:
SECTIONS = ["company_profile", "counterparties", "contracts", "practices"]
EXTRACTION = {
    "company_profile": {"facts": [
        {"section": "company_profile", "attribute": "regime", "value": "УСН", "evidence": "Мы на УСН"},
        {"section": "company_profile", "attribute": "usn_object", "value": "доходы", "evidence": "УСН доходы"},
        {"section": "company_profile", "attribute": "vat_payer", "value": "нет", "evidence": "НДС не платим"}],
        "section_statuses": []},
    "counterparties": {"facts": [
        {"section": "counterparties", "name": "Ромашка", "status": "active", "role": "поставщик",
         "evidence": "Работаем с Ромашкой как поставщик"}], "section_statuses": []},
    "contracts": {"facts": [
        {"section": "contracts", "name": "договор поставки с Ромашкой", "status": "active",
         "amount": 150000, "payment": "аванс", "evidence": "договор поставки на 150 тысяч с авансом"}],
        "section_statuses": []},
    "practices": {"facts": [
        {"section": "practices", "attribute": "cash_settlements", "value": "да", "evidence": "Расчёты наличными"}],
        "section_statuses": []},
}

def responder(messages, meta):
    t = meta.get("task")
    if t == "classify_sections":
        return json.dumps({"sections": SECTIONS})
    if t in ("extract", "extract_repair"):
        return json.dumps(EXTRACTION.get(meta.get("section"), {"facts": [], "section_statuses": []}))
    if t == "classify_query":
        return json.dumps({"query_types": ["deduction", "posting"], "subject": "self"})
    if t == "answer":
        if "Можно оплатить" in messages[-1]["content"]:
            return "Можно оплатить наличными и принять расходы на аренду."
        return "Понял, запомнил профиль компании. Чем помочь?"
    return "{}"

assistant = AccountantAssistant(ScriptedLLM(responder), store=MemoryStore())
session = assistant.session("demo", "d1", documents=load_bank_statement()[:5])

In [ ]:
turn1 = ("Мы на УСН доходы, НДС не платим. Работаем с Ромашкой как поставщик, "
         "договор поставки на 150 тысяч с авансом. Расчёты наличными.")
print("Бухгалтер:", turn1)
print("Ассистент:", session.say(turn1))
result = session.close()
print("изменения памяти:", result.summary()["changes"])
for f in assistant.store.list_facts("demo"):
    print(f"  {f.section.value:18} {f.key:35} {f.status:12} {f.data}")

In [ ]:
turn2 = "Можно оплатить Ромашке наличными 150 тысяч и принять расходы на аренду?"
print("Бухгалтер:", turn2)
session.say(turn2)
reply = session.replies[-1]
print("\n-- контекст (фрагмент) --\n", reply.read.bundle.render()[:900])
print("\n-- итоговый ответ (предупреждения дописаны кодом) --\n", reply.text)

## Оценка на dev/golden (без GPU)

Эталонные ответы extractor-а прогоняются через write-path; требуется 100% совпадение памяти.

In [ ]:
norm = Normalizer()
dev = run_devset(load_jsonl("data/dev_dialogs.jsonl"), norm)
print("dev:", dev["passed"], "/", dev["total"])
gold = run_golden(load_jsonl("data/golden/scenarios.jsonl"), norm)
print("golden clean:", gold["clean"], "/", gold["total"], "memory:", gold["memory"])
q = eval_query_classifier(load_jsonl("data/golden/queries.jsonl"))
print("baseline классификатор запросов:", q["accuracy"])

## Вывод

Ядро детерминировано и проверяемо: память, выборка контекста и проверка соответствия работают без модели. LLM-шаги оцениваются отдельно в `02_llm_pipeline.ipynb`.